In [29]:
from __future__ import print_function
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from base64 import urlsafe_b64decode
import re
import webbrowser
import time
import pandas as pd
import json
import unidecode
import dateutil.parser as dparser
import datetime

In [2]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

In [3]:
def get_creds(credentials_path, token_path, scopes):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, scopes)

    if not creds or not creds.valid:
        try:
            creds.refresh(Request())
        except:
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, scopes)
            creds = flow.run_local_server(port=0)

        with open(token_path, 'w') as token:
            token.write(creds.to_json())

    return creds

In [4]:
def read_message_subject(msg):
    payload = msg['payload']
    headers = payload.get("headers")
    if headers:
        for header in headers:
            name = header.get("name")
            value = header.get("value")
            if name.lower() == "subject":
                return value
    return None

In [5]:
def read_message_text(msg):
    payload = msg['payload']
    parts = payload.get("parts")
    part = parts[0]
    body = part.get("body")
    data = body.get("data")
    text = urlsafe_b64decode(data).decode()
    return text

In [6]:
def find_link(text):
    return re.search("(?P<url>https?://[^\s]+)", text).group("url")[:-1]

In [7]:
def next_creds(creds_and_tokens_dict, idx):
    idx = idx % len(creds_and_tokens_dict)
    return creds_and_tokens_dict[idx]['creds'], creds_and_tokens_dict[idx]['token']

In [8]:
def array_to_time_intervals(arr):
    time_intervals = [pd.Interval(pd.Timestamp(dparser.parse(i[0])), pd.Timestamp(dparser.parse(i[1]))) for i in arr]
    return time_intervals

In [33]:
def find_time_interval(txt):
    line_of_time = unidecode.unidecode(txt.split('EST')[1]).split(',')
    date = line_of_time[1]
    hours = line_of_time[2].split(' ')
    start = dparser.parse(date + ' ' + hours[3] + ' ' + hours[4], fuzzy=True)
    end = dparser.parse(date + ' ' + hours[6] + ' ' + hours[7], fuzzy=True)
    if start > end:
        end = end + datetime.timedelta(days=1)
    return pd.Interval(pd.Timestamp(start), pd.Timestamp(end))

In [10]:
# credentials_path = r'C:\Users\Fatemeh\Desktop\paper_shift_transfer\credentials.json'

credentials_path = '/home/matin/paper_gmail_shift_transfer/credentials/credentials_mtn.json'
token_path = '/home/matin/paper_gmail_shift_transfer/tokens/token_mtn.json'
scopes = ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/gmail.modify']
creds = get_creds(credentials_path, token_path, scopes)

service = build('gmail', 'v1', credentials=creds)

make_read_body = {"addLabelIds": [], "removeLabelIds": ['UNREAD']}

In [21]:
import inspect
from pprint import pprint
try:
    messages = service.users().messages().list(userId='me', maxResults=3, labelIds='UNREAD').execute()
    print(messages)
except Exception as e:
    error = e
    print(e)
    # pprint(inspect.getmembers(e))
    pprint(inspect.getmembers(error))
    print(error.content.decode('utf8').replace("'", '"'))

{'messages': [{'id': '1862cd2258f5b5d4', 'threadId': '1862c7d2c95bbd10'}, {'id': '1862ccff78ff2a3b', 'threadId': '1862c7d2c95bbd10'}, {'id': '185f4c38a68c90b8', 'threadId': '185f4c38a68c90b8'}], 'nextPageToken': '08841357642728318435', 'resultSizeEstimate': 201}


In [ ]:
f = open('/home/matin/paper_gmail_shift_transfer/unable_times.json')
arr = json.load(f)

In [274]:
scopes = ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/gmail.modify']

# credentials_path = r'C:\Users\Fatemeh\Desktop\paper_shift_transfer\credentials_mtn.json'
credentials1_path = 'credentials_mtn.json'
token1_path = 'token_mtn.json'
credentials2_path = 'credentials_fati.json'
token2_path = 'token_fati.json'

creds = get_creds(credentials_path, token_path, scopes)

service = build('gmail', 'v1', credentials=creds)

creds_and_tokens_path = [{'creds': credentials1_path, 'token':token1_path},
                         {'creds':credentials2_path, 'token':token2_path}]


In [277]:
pprint(service.users().labels().list(userId='me').execute())

{'labels': [{'id': 'CHAT',
             'labelListVisibility': 'labelHide',
             'messageListVisibility': 'hide',
             'name': 'CHAT',
             'type': 'system'},
            {'id': 'SENT', 'name': 'SENT', 'type': 'system'},
            {'id': 'INBOX', 'name': 'INBOX', 'type': 'system'},
            {'id': 'IMPORTANT',
             'labelListVisibility': 'labelHide',
             'messageListVisibility': 'hide',
             'name': 'IMPORTANT',
             'type': 'system'},
            {'id': 'TRASH',
             'labelListVisibility': 'labelHide',
             'messageListVisibility': 'hide',
             'name': 'TRASH',
             'type': 'system'},
            {'id': 'DRAFT', 'name': 'DRAFT', 'type': 'system'},
            {'id': 'SPAM',
             'labelListVisibility': 'labelHide',
             'messageListVisibility': 'hide',
             'name': 'SPAM',
             'type': 'system'},
            {'id': 'CATEGORY_FORUMS',
             'labelListVisib

In [12]:
messages = service.users().messages().list(userId='me', maxResults=1, labelIds='UNREAD').execute()
messages

{'messages': [{'id': '1864668306b805ac', 'threadId': '1864668306b805ac'}],
 'nextPageToken': '10157710542526310770',
 'resultSizeEstimate': 201}

In [13]:
messages = service.users().messages().list(userId='me', maxResults=1, labelIds='UNREAD').execute()
message_id = messages['messages'][0]['id']
msg = service.users().messages().get(userId='me', id=message_id, format='full').execute()
read_message_subject(msg)

'Shift Transfer Request'

In [34]:
txt = read_message_text(msg)

find_time_interval(txt)

Interval('2023-02-12 20:00:00', '2023-02-13', closed='right')

In [32]:
end + datetime.timedelta(days=1), end

(datetime.datetime(2023, 2, 13, 0, 0), datetime.datetime(2023, 2, 12, 0, 0))

In [291]:
if __name__ == '__main__':
    scopes = ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/gmail.modify']
    # credentials_path = r'C:\Users\Fatemeh\Desktop\paper_shift_transfer\credentials_mtn.json'
    credentials1_path = '/home/matin/paper_gmail_shift_transfer/credentials/credentials_mtn.json'
    token1_path = '/home/matin/paper_gmail_shift_transfer/tokens/token_mtn.json'
    credentials2_path = '/home/matin/paper_gmail_shift_transfer/credentials/credentials_fati.json'
    token2_path = '/home/matin/paper_gmail_shift_transfer/tokens/token_fati.json'

    creds_and_tokens_path = [{'creds': credentials1_path, 'token':token1_path},
                             {'creds':credentials2_path, 'token':token2_path}]

    creds_index = 0
    credentials_path = creds_and_tokens_path[creds_index]['creds']
    token_path = creds_and_tokens_path[creds_index]['token']

    creds = get_creds(credentials_path, token_path, scopes)
    service = build('gmail', 'v1', credentials=creds)
    make_read_body = {"addLabelIds": [], "removeLabelIds": ['UNREAD']}

    unable_path = '/home/matin/paper_gmail_shift_transfer/unable_times.json'
    unable_file = open(unable_path)
    unable_times = array_to_time_intervals(json.load(unable_file))


    while True:
        try:
            messages = service.users().messages().list(userId='me', maxResults=1, labelIds='UNREAD').execute()
            message_id = messages['messages'][0]['id']
            msg = service.users().messages().get(userId='me', id=message_id, format='full').execute()
            if read_message_subject(msg) == 'Shift Transfer Request':
                text = read_message_text(msg)
                request_interval = find_time_interval(text)
                able = True
                for interval in unable_times:
                    if interval.overlaps(request_interval):
                        able = False
                if able:
                    link = find_link(text)
                    webbrowser.open(link)
                    print(f'{bcolors.OKBLUE}{time.ctime(time.time())}')
                    print(link)
                    print(f'{bcolors.WARNING}-----------------------------------')
                else:
                    print(f'{bcolors.OKBLUE}{time.ctime(time.time())}')
                    print('Rejected - Bad timing')
                    print(f'{bcolors.WARNING}XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
                service.users().messages().modify(userId='me', id=message_id, body=make_read_body).execute()
        except Exception as e:
                print(f'{bcolors.FAIL}{time.ctime(time.time())}')
                print(f'{bcolors.FAIL}{e}')
                print(f'{bcolors.WARNING}-----------------------------------')

                if e.args[0] == 'invalid_grant: Token has been expired or revoked.':
                    creds = get_creds(credentials_path, token_path, scopes)
                    service = build('gmail', 'v1', credentials=creds)
                elif 'User-rate limit exceeded' in e.reason:
                    creds_index += 1
                    credentials_path, token_path = next_creds(creds_and_tokens_path, creds_index)
                    creds = get_creds(credentials_path, token_path, scopes)
                    service = build('gmail', 'v1', credentials=creds)
                    print('token and creds were change due to user-rate limit')

        time.sleep(0.1)


Tue Feb  7 19:20:41 2023
Rejected - Bad timing
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Tue Feb  7 19:22:39 2023
https://mandrillapp.com/track/click/30952325/app.paper.co?p=eyJzIjoicXhfY0VwN2tJdWphcVVjWGROMzZGODlVWVprIiwidiI6MSwicCI6IntcInVcIjozMDk1MjMyNSxcInZcIjoxLFwidXJsXCI6XCJodHRwczpcXFwvXFxcL2FwcC5wYXBlci5jb1xcXC90dXRvclxcXC9zY2hlZHVsZVxcXC8zMjYwNFxcXC92YWxpZGF0ZS10cmFuc2ZlclwiLFwiaWRcIjpcIjI0ZDY5ZjZmY2M0ZjRiNThiMWViYjNhYjEyZTAxYWE1XCIsXCJ1cmxfaWRzXCI6W1wiMmE5NmU0OWM0YTNjY2MzNjdjM2VlYjA1MDc5ZGNmY2Q2YjJkYmM3NFwiXX0ifQ
-----------------------------------


KeyboardInterrupt: 

In [293]:
url = 'https://mandrillapp.com/track/click/30952325/app.paper.co?p=eyJzIjoicXhfY0VwN2tJdWphcVVjWGROMzZGODlVWVprIiwidiI6MSwicCI6IntcInVcIjozMDk1MjMyNSxcInZcIjoxLFwidXJsXCI6XCJodHRwczpcXFwvXFxcL2FwcC5wYXBlci5jb1xcXC90dXRvclxcXC9zY2hlZHVsZVxcXC8zMjYwNFxcXC92YWxpZGF0ZS10cmFuc2ZlclwiLFwiaWRcIjpcIjI0ZDY5ZjZmY2M0ZjRiNThiMWViYjNhYjEyZTAxYWE1XCIsXCJ1cmxfaWRzXCI6W1wiMmE5NmU0OWM0YTNjY2MzNjdjM2VlYjA1MDc5ZGNmY2Q2YjJkYmM3NFwiXX0ifQ'

True